# Dataset B GPU-00 — Colab environment probe

This notebook verifies the Colab GPU environment and persistent-storage path only. It does not read Dataset B, run preprocessing, import a model wrapper, train a model, access Validation/Test, or produce RQ1 evidence. Run it top-to-bottom after reviewing the parameter cell.

## 1. Colab setup

Select runtime version `2026.07`, enable a GPU accelerator, and make the repository available at `/content/DAP391m_GR12`. Leave High-RAM disabled. Do not start SSH. See `docs/handbook/archive/colab_environment_superseded.md` for the superseded-environment redirect.

In [ ]:
from pathlib import Path

INSTALL_GPU_ENV = False
EXPECTED_PYTHON = '3.12.13'
EXPECTED_TORCH_BASE = '2.14.0'
TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu126'
REPO_ROOT = Path('/content/DAP391m_GR12')
RUNTIME_CONFIG = REPO_ROOT / 'config' / 'dataset_b' / 'runtime_colab_gpu_v1.yaml'
DRIVE_MOUNT = Path('/content/drive')
EVIDENCE_DIR = DRIVE_MOUNT / 'MyDrive' / 'DAP391m_GR12_remote' / 'gpu00'
EVIDENCE_NAME = 'gpu00_environment.json'


## 2. Optional GPU probe dependencies

The repository `requirements.txt` is CPU-oriented and is not used here. Set `INSTALL_GPU_ENV = True` only when the selected PyTorch CUDA build must be installed. If Colab requests a runtime restart, restart and rerun from the top with installation disabled.

In [ ]:
import subprocess
import sys

if INSTALL_GPU_ENV:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--upgrade', 'torch==2.14.0', '--index-url', TORCH_INDEX_URL],
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--upgrade', 'pytabkit==1.7.3', 'PyYAML==6.0.3'],
        check=True,
    )
    print('Installation completed. Restart the runtime if Colab requests it, then rerun from the top.')
else:
    print('Dependency installation skipped; the current runtime will be verified as-is.')


## 3. Mount persistent storage

In [ ]:
try:
    from google.colab import drive
except ModuleNotFoundError as exc:
    raise RuntimeError('GPU-00 must run inside Google Colab.') from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)


## 4. Verify repository and runtime

In [ ]:
from datetime import datetime, timezone
import hashlib
from importlib.metadata import PackageNotFoundError, version
import json
import math
import platform

import torch

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def package_version(name: str):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

require(REPO_ROOT.is_dir(), f'Repository is missing: {REPO_ROOT}')
require(RUNTIME_CONFIG.is_file(), f'Runtime config is missing: {RUNTIME_CONFIG}')
python_version = platform.python_version()
torch_version = version('torch')
torch_base = torch_version.split('+', maxsplit=1)[0]
require(python_version == EXPECTED_PYTHON, f'Expected Python {EXPECTED_PYTHON}, found {python_version}.')
require(torch_base == EXPECTED_TORCH_BASE, f'Expected PyTorch {EXPECTED_TORCH_BASE}, found {torch_version}.')
require(torch.version.cuda is not None, 'The installed PyTorch build does not expose CUDA.')
require(torch.cuda.is_available(), 'CUDA is not available to PyTorch.')
require(torch.cuda.device_count() >= 1, 'No CUDA device is visible.')

git_commit = subprocess.run(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
git_status = subprocess.run(
    ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
    check=True, capture_output=True, text=True,
).stdout.strip()
require(not git_status, 'Repository working tree must be clean for GPU-00 evidence.')

nvidia_smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader,nounits'],
    check=True, capture_output=True, text=True,
).stdout.strip().splitlines()
device_index = 0
device_name = torch.cuda.get_device_name(device_index)
compute_capability = list(torch.cuda.get_device_capability(device_index))
total_memory_bytes = int(torch.cuda.get_device_properties(device_index).total_memory)
torch.use_deterministic_algorithms(True)
left = torch.arange(1, 1025, device='cuda', dtype=torch.float32)
probe_value = float((left.square().mean()).detach().cpu())
require(math.isfinite(probe_value), 'CUDA tensor probe produced a non-finite value.')

evidence = {
    'schema_version': 1,
    'stage': 'gpu_00_environment_probe',
    'status': 'PASS',
    'probed_at_utc': datetime.now(timezone.utc).isoformat(),
    'repository': {
        'path': str(REPO_ROOT),
        'git_commit': git_commit,
        'working_tree_clean': True,
    },
    'runtime_config': {
        'path': str(RUNTIME_CONFIG.relative_to(REPO_ROOT)),
        'sha256': sha256_file(RUNTIME_CONFIG),
    },
    'environment': {
        'python': python_version,
        'torch': torch_version,
        'torch_cuda': torch.version.cuda,
        'cuda_available': True,
        'cuda_device_count': torch.cuda.device_count(),
        'gpu_name': device_name,
        'compute_capability': compute_capability,
        'gpu_total_memory_bytes': total_memory_bytes,
        'nvidia_smi': nvidia_smi,
        'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
        'packages': {
            'numpy': package_version('numpy'),
            'pandas': package_version('pandas'),
            'scipy': package_version('scipy'),
            'scikit-learn': package_version('scikit-learn'),
            'pytabkit': package_version('pytabkit'),
            'PyYAML': package_version('PyYAML'),
        },
    },
    'checks': {
        'exact_python_version': True,
        'exact_torch_release': True,
        'cuda_build_present': True,
        'cuda_available': True,
        'gpu_identity_and_driver': bool(device_name and nvidia_smi),
        'finite_cuda_tensor_operation': True,
        'repository_commit_identity': bool(git_commit),
        'repository_working_tree_clean': True,
        'runtime_config_hash': True,
        'dataset_read': False,
        'validation_read': False,
        'test_read': False,
        'model_training': False,
    },
    'probe': {'cuda_tensor_mean_square': probe_value},
}

evidence


## 5. Publish and verify GPU-00 evidence

In [ ]:
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
final_path = EVIDENCE_DIR / EVIDENCE_NAME
temporary_path = EVIDENCE_DIR / f'.{EVIDENCE_NAME}.tmp'
payload = json.dumps(evidence, ensure_ascii=False, indent=2)
temporary_path.write_text(payload, encoding='utf-8')
read_back = json.loads(temporary_path.read_text(encoding='utf-8'))
require(read_back == evidence, 'Google Drive write/read verification failed.')
evidence['checks']['google_drive_atomic_write_read'] = True
payload = json.dumps(evidence, ensure_ascii=False, indent=2)
temporary_path.write_text(payload, encoding='utf-8')
temporary_path.replace(final_path)
require(json.loads(final_path.read_text(encoding='utf-8')) == evidence, 'Published evidence differs after rename.')
print({'status': evidence['status'], 'evidence': str(final_path), 'gpu': device_name, 'torch': torch_version})


## 6. Handoff

GPU-00 is not complete until `gpu00_environment.json` is independently reviewed. A PASS permits GPU-01 to implement CUDA device handling and resumable checkpoint state in the plain-MLP wrapper. It does not permit model training by itself.